# 02 — Stochastic Debt Sustainability Analysis

This notebook builds a stochastic debt sustainability analysis (DSA) for France.

It separates two accounting layers:

1. **AFT negotiable French State debt** — detailed refinancing and interest-cost transmission.
2. **French general-government debt** — broader debt/GDP sustainability.

The AFT portfolio is not treated as the whole French public-debt stock. Instead, the AFT engine estimates the **incremental financing-cost effect of market deviations** relative to a central AFT path.

The key bridge is:

\[
\Delta IC_t^{AFT}
=
IC_t^{AFT,stochastic}
-
IC_t^{AFT,central}
\]

The general-government central path is calibrated to the IMF 2026 Article IV baseline through 2031. The stochastic general-government path then applies the central GG effective rate to the stochastic debt stock and adds only the incremental AFT financing-cost shock.

Stages:

1. Load and validate the AFT portfolio.
2. Run standalone deterministic AFT yield sensitivities.
3. Calibrate stochastic processes from historical data.
4. Build an IMF-consistent 2026–2041 central scenario.
5. Generate a robust stochastic macro/market path.
6. Run both central and stochastic AFT portfolios.
7. Build and validate the central general-government path.
8. Transfer only the AFT cost deviation from central into the stochastic GG DSA.

In [ ]:
from pathlib import Path
from io import StringIO
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from sklearn.covariance import LedoitWolf, MinCovDet
from scipy.stats import chi2

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

from src.models.debt_portfolio import (
    load_portfolio,
    portfolio_summary,
    maturity_schedule,
    total_debt,
    annual_coupon_expense,
    portfolio_coupon_rate,
    simulate_portfolio_path,
    curve_from_30y_anchor,
)

RANDOM_SEED = 42
print("Project root:", PROJECT_ROOT)

## 1. Load the validated AFT portfolio

In [ ]:
SNAPSHOT_DATE = pd.Timestamp("2026-10-03")

PORTFOLIO_PATH = (
    PROCESSED_DIR
    / f"aft_portfolio_{SNAPSHOT_DATE.date()}.csv"
)

portfolio = load_portfolio(
    PORTFOLIO_PATH,
    snapshot_date=SNAPSHOT_DATE,
)

print(f"Securities loaded: {len(portfolio):,}")
print(f"AFT indexed liability: €{total_debt(portfolio) / 1e12:.3f}tn")
print(f"Annual coupon cash: €{annual_coupon_expense(portfolio) / 1e9:.1f}bn")
print(f"Portfolio coupon rate: {portfolio_coupon_rate(portfolio):.2%}")

In [ ]:
portfolio_summary(portfolio)

In [ ]:
maturity_schedule(portfolio).head(15)

## 2. General-government starting calibration

In [ ]:
# ============================================================
# IMF-consistent starting point: end-2026
# ============================================================

BASELINE = {
    "initial_gdp_eur": 3_056e9,
    "initial_debt_gdp": 1.185,
    "initial_primary_balance_gdp": -0.027,
    "initial_interest_gdp": 0.025,

    # Used only by the standalone deterministic AFT section
    "real_growth": 0.009,
    "inflation": 0.017,

    "start_year": 2027,
    "horizon_years": 15,
}

BASELINE["nominal_growth"] = (
    (1 + BASELINE["real_growth"])
    * (1 + BASELINE["inflation"])
    - 1
)

INITIAL_GG_DEBT_EUR = (
    BASELINE["initial_gdp_eur"]
    * BASELINE["initial_debt_gdp"]
)

print(f"Initial general-government GDP: €{BASELINE['initial_gdp_eur'] / 1e12:.3f}tn")
print(f"Initial general-government debt: €{INITIAL_GG_DEBT_EUR / 1e12:.3f}tn")
print(f"Initial debt/GDP: {BASELINE['initial_debt_gdp']:.1%}")
print(f"Initial primary balance/GDP: {BASELINE['initial_primary_balance_gdp']:.1%}")
print(f"2026 interest/GDP: {BASELINE['initial_interest_gdp']:.1%}")

## 3. Deterministic AFT baseline and 7% stress

In [ ]:
BASELINE_OAT_30Y = 0.055

NOMINAL_CURVE_OFFSETS_TO_30Y = {
    1.0:  -0.0150,
    2.0:  -0.0125,
    5.0:  -0.0100,
    10.0: -0.0060,
    15.0: -0.0035,
    30.0:  0.0000,
    50.0:  0.0015,
}

baseline_nominal_curve = curve_from_30y_anchor(
    oat_30y_yield=BASELINE_OAT_30Y,
    offsets_to_30y=NOMINAL_CURVE_OFFSETS_TO_30Y,
)

baseline_real_euro_curve = {
    5.0: 0.020,
    10.0: 0.025,
    20.0: 0.028,
    30.0: 0.030,
}

baseline_real_fr_curve = {
    5.0: 0.021,
    10.0: 0.026,
    20.0: 0.029,
    30.0: 0.031,
}

baseline_market_curves = {
    "nominal": baseline_nominal_curve,
    "real_euro": baseline_real_euro_curve,
    "real_fr": baseline_real_fr_curve,
}

ISSUANCE_PLAN = [
    {"security_type": "BTF",   "maturity_years": 1.0,  "weight": 0.10},
    {"security_type": "OAT",   "maturity_years": 2.0,  "weight": 0.10},
    {"security_type": "OAT",   "maturity_years": 5.0,  "weight": 0.18},
    {"security_type": "OAT",   "maturity_years": 10.0, "weight": 0.27},
    {"security_type": "OATEI", "maturity_years": 10.0, "weight": 0.05},
    {"security_type": "OATI",  "maturity_years": 10.0, "weight": 0.03},
    {"security_type": "OAT",   "maturity_years": 15.0, "weight": 0.10},
    {"security_type": "OAT",   "maturity_years": 30.0, "weight": 0.12},
    {"security_type": "OAT",   "maturity_years": 50.0, "weight": 0.05},
]

assert np.isclose(
    sum(x["weight"] for x in ISSUANCE_PLAN),
    1.0,
)

In [ ]:
aft_baseline_path, aft_baseline_final = simulate_portfolio_path(
    initial_portfolio=portfolio,
    initial_gdp_eur=BASELINE["initial_gdp_eur"],
    start_year=BASELINE["start_year"],
    years=BASELINE["horizon_years"],
    nominal_growth=BASELINE["nominal_growth"],
    primary_balance_ratio=BASELINE["initial_primary_balance_gdp"],
    market_curves=baseline_market_curves,
    issuance_plan=ISSUANCE_PLAN,
    french_inflation=BASELINE["inflation"],
    euro_inflation=BASELINE["inflation"],
)

aft_baseline_path[
    [
        "year",
        "weighted_market_issue_rate",
        "effective_financing_rate",
        "coupon_cash_eur",
        "total_financing_cost_eur",
        "portfolio_debt_end_eur",
    ]
]

In [ ]:
STRESS_OAT_30Y = 0.070

stress_nominal_curve = curve_from_30y_anchor(
    oat_30y_yield=STRESS_OAT_30Y,
    offsets_to_30y=NOMINAL_CURVE_OFFSETS_TO_30Y,
)

stress_market_curves = {
    "nominal": stress_nominal_curve,
    "real_euro": baseline_real_euro_curve,
    "real_fr": baseline_real_fr_curve,
}

aft_stress_path, aft_stress_final = simulate_portfolio_path(
    initial_portfolio=portfolio,
    initial_gdp_eur=BASELINE["initial_gdp_eur"],
    start_year=BASELINE["start_year"],
    years=BASELINE["horizon_years"],
    nominal_growth=BASELINE["nominal_growth"],
    primary_balance_ratio=BASELINE["initial_primary_balance_gdp"],
    market_curves=stress_market_curves,
    issuance_plan=ISSUANCE_PLAN,
    french_inflation=BASELINE["inflation"],
    euro_inflation=BASELINE["inflation"],
)

aft_comparison = pd.DataFrame({
    "year": aft_baseline_path["year"],
    "baseline_effective_rate": aft_baseline_path["effective_financing_rate"],
    "stress_effective_rate": aft_stress_path["effective_financing_rate"],
    "baseline_financing_cost_eur": aft_baseline_path["total_financing_cost_eur"],
    "stress_financing_cost_eur": aft_stress_path["total_financing_cost_eur"],
})

aft_comparison["interest_cost_shock_eur"] = (
    aft_comparison["stress_financing_cost_eur"]
    - aft_comparison["baseline_financing_cost_eur"]
)

aft_comparison["interest_cost_shock_bn"] = (
    aft_comparison["interest_cost_shock_eur"] / 1e9
)

aft_comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    aft_baseline_path["year"],
    aft_baseline_path["effective_financing_rate"] * 100,
    label="5.5% 30Y OAT baseline",
)

ax.plot(
    aft_stress_path["year"],
    aft_stress_path["effective_financing_rate"] * 100,
    label="7.0% 30Y OAT stress",
)

ax.set_title("AFT Portfolio Effective Financing Rate")
ax.set_xlabel("Year")
ax.set_ylabel("%")
ax.legend()
ax.grid(alpha=0.25)

plt.show()

## 4. Standalone deterministic AFT sensitivity

The 5.5% and 7.0% 30Y OAT scenarios above remain useful as standalone funding-cost sensitivities.

They are **not** the reference scenario for the stochastic bridge. The stochastic bridge later compares the stochastic AFT path against a separate time-varying **central AFT path**.

In [ ]:
aft_comparison[
    [
        "year",
        "baseline_effective_rate",
        "stress_effective_rate",
        "interest_cost_shock_bn",
    ]
]

In [ ]:
YEARS = aft_baseline_path["year"].astype(int).tolist()

assert YEARS == list(range(2027, 2042))

print("Simulation horizon:", YEARS[0], "to", YEARS[-1])

# 5. Historical calibration dataset

The stochastic model is calibrated on annual observations from **1999–2025**.

Variables:

- French real GDP growth
- French HICP inflation
- French general-government primary balance / GDP
- German 10Y Bund yield
- France–Germany 10Y sovereign spread

In [ ]:
ECB_CACHE_DIR = RAW_DIR / "ecb"

def create_ecb_session():
    retry_strategy = Retry(
        total=6,
        connect=6,
        read=6,
        status=6,
        backoff_factor=1.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET"]),
        respect_retry_after_header=True,
    )

    adapter = HTTPAdapter(max_retries=retry_strategy)
    session = requests.Session()
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    session.headers.update({
        "Accept": "text/csv",
        "Accept-Encoding": "gzip, deflate",
        "User-Agent": "french-oat-model/1.0",
    })

    return session

ECB_SESSION = create_ecb_session()


def fetch_ecb_series(
    series_key,
    start_period="1999-01",
    end_period="2025-12",
    cache_dir=ECB_CACHE_DIR,
    force_refresh=False,
):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    safe_key = (
        series_key
        .replace(".", "_")
        .replace("+", "_")
    )

    cache_path = (
        cache_dir
        / f"ecb_IRS_{safe_key}_{start_period}_{end_period}.csv"
    )

    if cache_path.exists() and not force_refresh:
        print(f"Loading cached ECB data:\n{cache_path}")
        return pd.read_csv(
            cache_path,
            parse_dates=["TIME_PERIOD"],
        )

    url = (
        "https://data-api.ecb.europa.eu/"
        f"service/data/IRS/{series_key}"
    )

    params = {
        "startPeriod": start_period,
        "endPeriod": end_period,
        "format": "csvdata",
        "detail": "dataonly",
    }

    response = ECB_SESSION.get(
        url,
        params=params,
        timeout=(10, 180),
    )
    response.raise_for_status()

    df = pd.read_csv(StringIO(response.text))

    required_columns = {"TIME_PERIOD", "OBS_VALUE"}
    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(
            f"ECB response missing columns: {missing}"
        )

    df["TIME_PERIOD"] = pd.to_datetime(
        df["TIME_PERIOD"],
        errors="coerce",
    )

    df["OBS_VALUE"] = pd.to_numeric(
        df["OBS_VALUE"],
        errors="coerce",
    )

    df = (
        df[["TIME_PERIOD", "OBS_VALUE"]]
        .dropna()
        .sort_values("TIME_PERIOD")
        .reset_index(drop=True)
    )

    df.to_csv(cache_path, index=False)
    return df

In [ ]:
FRANCE_10Y_KEY = "M.FR.L.L40.CI.0000.EUR.N.Z"
GERMANY_10Y_KEY = "M.DE.L.L40.CI.0000.EUR.N.Z"

france_10y_monthly = fetch_ecb_series(
    FRANCE_10Y_KEY,
    start_period="1999-01",
    end_period="2025-12",
)

germany_10y_monthly = fetch_ecb_series(
    GERMANY_10Y_KEY,
    start_period="1999-01",
    end_period="2025-12",
)

def monthly_rate_to_annual(
    df,
    column_name,
):
    result = df.copy()
    result["year"] = result["TIME_PERIOD"].dt.year
    result[column_name] = (
        pd.to_numeric(
            result["OBS_VALUE"],
            errors="coerce",
        )
        / 100
    )

    return (
        result[["year", column_name]]
        .dropna()
        .groupby("year", as_index=False)[column_name]
        .mean()
    )

france_10y_annual = monthly_rate_to_annual(
    france_10y_monthly,
    "france_10y",
)

germany_10y_annual = monthly_rate_to_annual(
    germany_10y_monthly,
    "bund_10y",
)

rates_history = (
    france_10y_annual
    .merge(
        germany_10y_annual,
        on="year",
        how="inner",
        validate="one_to_one",
    )
)

rates_history["france_spread_10y"] = (
    rates_history["france_10y"]
    - rates_history["bund_10y"]
)

rates_history = (
    rates_history[
        rates_history["year"].between(1999, 2025)
    ]
    .sort_values("year")
    .reset_index(drop=True)
)

rates_history

In [ ]:
EUROSTAT_CACHE_DIR = RAW_DIR / "eurostat"

def _jsonstat_category_labels(category_index):
    if isinstance(category_index, dict):
        return [
            label
            for label, _
            in sorted(
                category_index.items(),
                key=lambda x: x[1],
            )
        ]
    return list(category_index)


def fetch_eurostat_series(
    dataset_code,
    filters,
    value_name,
    cache_dir=EUROSTAT_CACHE_DIR,
    force_refresh=False,
):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    filter_string = "_".join(
        f"{key}-{value}"
        for key, value
        in sorted(filters.items())
    )

    cache_path = (
        cache_dir
        / f"{dataset_code}_{filter_string}.csv"
    )

    if cache_path.exists() and not force_refresh:
        print(f"Loading cached Eurostat data:\n{cache_path}")
        return pd.read_csv(cache_path)

    url = (
        "https://ec.europa.eu/"
        "eurostat/api/dissemination/"
        "statistics/1.0/data/"
        f"{dataset_code}"
    )

    params = {
        "lang": "EN",
        **filters,
    }

    response = requests.get(
        url,
        params=params,
        timeout=(10, 180),
    )
    response.raise_for_status()
    data = response.json()

    dimension_ids = data["id"]
    dimension_sizes = data["size"]

    if "time" not in dimension_ids:
        raise ValueError(
            "Eurostat response contains no time dimension."
        )

    time_pos = dimension_ids.index("time")

    for i, dimension in enumerate(dimension_ids):
        if dimension != "time" and dimension_sizes[i] != 1:
            raise ValueError(
                f"Dimension {dimension!r} was not reduced to one category."
            )

    years = _jsonstat_category_labels(
        data["dimension"]["time"]["category"]["index"]
    )

    values = data["value"]
    rows = []

    for time_coordinate, year in enumerate(years):
        coordinates = [0 for _ in dimension_ids]
        coordinates[time_pos] = time_coordinate

        flat_index = int(
            np.ravel_multi_index(
                tuple(coordinates),
                tuple(dimension_sizes),
                order="C",
            )
        )

        if isinstance(values, dict):
            value = values.get(str(flat_index), np.nan)
        else:
            value = (
                values[flat_index]
                if flat_index < len(values)
                else np.nan
            )

        rows.append({
            "year": int(year),
            value_name: value,
        })

    result = pd.DataFrame(rows)
    result[value_name] = pd.to_numeric(
        result[value_name],
        errors="coerce",
    )

    result.to_csv(cache_path, index=False)
    return result

In [ ]:
real_growth_history = fetch_eurostat_series(
    dataset_code="nama_10_gdp",
    filters={
        "freq": "A",
        "unit": "CLV_PCH_PRE",
        "na_item": "B1GQ",
        "geo": "FR",
    },
    value_name="real_growth",
)

real_growth_history["real_growth"] /= 100

real_growth_history = (
    real_growth_history[
        real_growth_history["year"].between(1999, 2025)
    ]
    .reset_index(drop=True)
)

inflation_history = fetch_eurostat_series(
    dataset_code="prc_hicp_aind",
    filters={
        "freq": "A",
        "unit": "RCH_A_AVG",
        "coicop": "CP00",
        "geo": "FR",
    },
    value_name="inflation",
)

inflation_history["inflation"] /= 100

inflation_history = (
    inflation_history[
        inflation_history["year"].between(1999, 2025)
    ]
    .reset_index(drop=True)
)

In [ ]:
DBNOMICS_CACHE_DIR = RAW_DIR / "ameco"

def fetch_ameco_primary_balance(
    cache_dir=DBNOMICS_CACHE_DIR,
    force_refresh=False,
):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    cache_path = (
        cache_dir
        / "france_primary_balance_gdp.csv"
    )

    if cache_path.exists() and not force_refresh:
        print(f"Loading cached AMECO data:\n{cache_path}")
        return pd.read_csv(cache_path)

    series_code = "FRA.1.0.319.0.UBLGI"

    url = (
        "https://api.db.nomics.world/"
        "v22/series/"
        f"AMECO/UBLGI/{series_code}"
    )

    response = requests.get(
        url,
        params={"observations": 1},
        timeout=(10, 180),
    )
    response.raise_for_status()

    payload = response.json()

    docs = (
        payload
        .get("series", {})
        .get("docs", [])
    )

    if not docs:
        raise ValueError(
            "DBnomics returned no AMECO observations."
        )

    series = docs[0]

    result = pd.DataFrame({
        "year": pd.to_numeric(
            series["period"],
            errors="coerce",
        ),
        "primary_balance_gdp": pd.to_numeric(
            series["value"],
            errors="coerce",
        ),
    })

    result = result.dropna().copy()
    result["year"] = result["year"].astype(int)
    result["primary_balance_gdp"] /= 100

    result = (
        result[
            result["year"].between(1999, 2025)
        ]
        .reset_index(drop=True)
    )

    result.to_csv(cache_path, index=False)
    return result

primary_balance_history = (
    fetch_ameco_primary_balance()
)

In [ ]:
macro_history = (
    real_growth_history
    .merge(
        inflation_history,
        on="year",
        how="inner",
    )
    .merge(
        primary_balance_history,
        on="year",
        how="inner",
    )
)

calibration_data = (
    macro_history
    .merge(
        rates_history[
            [
                "year",
                "bund_10y",
                "france_spread_10y",
            ]
        ],
        on="year",
        how="inner",
    )
)

CALIBRATION_START_YEAR = 1999
CALIBRATION_END_YEAR = 2025

CALIBRATION_VARIABLES = [
    "real_growth",
    "inflation",
    "primary_balance_gdp",
    "bund_10y",
    "france_spread_10y",
]

calibration_data = (
    calibration_data[
        calibration_data["year"].between(
            CALIBRATION_START_YEAR,
            CALIBRATION_END_YEAR,
        )
    ]
    .sort_values("year")
    .reset_index(drop=True)
)

expected_years = set(
    range(
        CALIBRATION_START_YEAR,
        CALIBRATION_END_YEAR + 1,
    )
)

missing_years = (
    expected_years
    - set(calibration_data["year"])
)

assert not missing_years, (
    f"Missing calibration years: {sorted(missing_years)}"
)

assert calibration_data["year"].is_unique

assert not (
    calibration_data[
        CALIBRATION_VARIABLES
    ]
    .isna()
    .any()
    .any()
)

assert len(calibration_data) == 27

print("Historical calibration dataset validation passed.")
calibration_data

## 6. AR(1) calibration and robust covariance

In [ ]:
def estimate_ar1(
    series,
    variable_name,
):
    x = (
        pd.Series(series)
        .dropna()
        .astype(float)
        .to_numpy()
    )

    if len(x) < 10:
        raise ValueError(
            f"Too few observations for {variable_name}: {len(x)}"
        )

    y = x[1:]
    x_lag = x[:-1]

    X = np.column_stack([
        np.ones(len(x_lag)),
        x_lag,
    ])

    beta = np.linalg.lstsq(
        X,
        y,
        rcond=None,
    )[0]

    intercept = float(beta[0])
    rho = float(beta[1])

    fitted = (
        intercept
        + rho * x_lag
    )

    residuals = (
        y
        - fitted
    )

    degrees_of_freedom = len(residuals) - 2

    innovation_std = float(
        np.sqrt(
            np.sum(residuals ** 2)
            / degrees_of_freedom
        )
    )

    if abs(rho) < 1.0:
        long_run_mean = (
            intercept
            / (1.0 - rho)
        )
    else:
        long_run_mean = np.nan

    return {
        "variable": variable_name,
        "n_observations": len(x),
        "intercept": intercept,
        "rho": rho,
        "long_run_mean": long_run_mean,
        "innovation_std": innovation_std,
        "fitted": fitted,
        "residuals": residuals,
    }


ar1_results = {
    variable: estimate_ar1(
        calibration_data[variable],
        variable,
    )
    for variable
    in CALIBRATION_VARIABLES
}

ar1_summary = pd.DataFrame([
    {
        "variable": variable,
        "long_run_mean_pct": (
            result["long_run_mean"] * 100
        ),
        "rho": result["rho"],
        "innovation_std_pct": (
            result["innovation_std"] * 100
        ),
    }
    for variable, result
    in ar1_results.items()
])

ar1_summary.round(3)

In [ ]:
RESIDUAL_YEARS = (
    calibration_data["year"]
    .iloc[1:]
    .to_numpy()
)

residual_data = pd.DataFrame({
    variable:
        ar1_results[variable]["residuals"]
    for variable
    in CALIBRATION_VARIABLES
})

residual_data.insert(
    0,
    "year",
    RESIDUAL_YEARS,
)

RESIDUAL_MATRIX = (
    residual_data[
        CALIBRATION_VARIABLES
    ]
    .to_numpy(dtype=float)
)

raw_residual_correlation = (
    residual_data[
        CALIBRATION_VARIABLES
    ]
    .corr()
)

raw_residual_correlation.round(3)

In [ ]:
standardized_residuals = (
    residual_data[
        CALIBRATION_VARIABLES
    ]
    - residual_data[
        CALIBRATION_VARIABLES
    ].mean()
) / residual_data[
    CALIBRATION_VARIABLES
].std(ddof=1)

lw = LedoitWolf().fit(
    standardized_residuals.values
)

shrunk_covariance = lw.covariance_

std = np.sqrt(
    np.diag(shrunk_covariance)
)

CALIBRATED_CORRELATION_MATRIX = (
    shrunk_covariance
    / np.outer(std, std)
)

correlation_df = pd.DataFrame(
    CALIBRATED_CORRELATION_MATRIX,
    index=CALIBRATION_VARIABLES,
    columns=CALIBRATION_VARIABLES,
)

correlation_df.round(3)

In [ ]:
PERSISTENCE = {
    variable:
        ar1_results[variable]["rho"]
    for variable
    in CALIBRATION_VARIABLES
}

SHOCK_STD = {
    variable:
        ar1_results[variable]["innovation_std"]
    for variable
    in CALIBRATION_VARIABLES
}

shock_std_vector = np.array([
    SHOCK_STD[variable]
    for variable
    in CALIBRATION_VARIABLES
])

CALIBRATED_COVARIANCE_MATRIX = (
    np.outer(
        shock_std_vector,
        shock_std_vector,
    )
    * CALIBRATED_CORRELATION_MATRIX
)

assert (
    np.linalg.eigvalsh(
        CALIBRATED_COVARIANCE_MATRIX
    ) > 0
).all()

print(
    "Ledoit-Wolf covariance matrix is positive definite."
)

In [ ]:
# Preferred ordinary-times covariance:
# Minimum Covariance Determinant (robust to crisis outliers).

X = residual_data[
    CALIBRATION_VARIABLES
].to_numpy(dtype=float)

mcd = MinCovDet(
    random_state=RANDOM_SEED
).fit(X)

ROBUST_RESIDUAL_MEAN = (
    mcd.location_
)

ROBUST_COVARIANCE_MATRIX = (
    mcd.covariance_
)

robust_eigenvalues = (
    np.linalg.eigvalsh(
        ROBUST_COVARIANCE_MATRIX
    )
)

assert (
    robust_eigenvalues > 0
).all()

robust_std = np.sqrt(
    np.diag(
        ROBUST_COVARIANCE_MATRIX
    )
)

volatility_comparison = pd.DataFrame({
    "variable":
        CALIBRATION_VARIABLES,

    "full_sample_std_pct":
        [
            SHOCK_STD[x] * 100
            for x
            in CALIBRATION_VARIABLES
        ],

    "robust_std_pct":
        robust_std * 100,
})

volatility_comparison.round(3)

In [ ]:
robust_distance_squared = (
    mcd.mahalanobis(X)
)

OUTLIER_CONFIDENCE = 0.975

outlier_threshold = chi2.ppf(
    OUTLIER_CONFIDENCE,
    df=len(CALIBRATION_VARIABLES),
)

residual_diagnostics = (
    residual_data.copy()
)

residual_diagnostics[
    "robust_mahalanobis_sq"
] = robust_distance_squared

residual_diagnostics[
    "stress_observation"
] = (
    residual_diagnostics[
        "robust_mahalanobis_sq"
    ]
    > outlier_threshold
)

residual_diagnostics[
    [
        "year",
        "robust_mahalanobis_sq",
        "stress_observation",
    ]
].sort_values(
    "robust_mahalanobis_sq",
    ascending=False,
)

## 7. Time-varying central scenario

The central path is anchored to the IMF 2026 Article IV baseline through 2031:

- nominal GDP;
- real GDP growth;
- CPI;
- GDP deflator;
- primary balance;
- overall balance;
- public debt.

After 2031, macro/fiscal variables are held at transparent terminal assumptions. The market central path continues to follow the conditional expectation of the calibrated AR(1) processes.

The market path feeds the AFT refinancing engine; the IMF fiscal path validates the broader general-government DSA.

In [ ]:
# Latest market state from 2026 ECB observations.

france_10y_current = fetch_ecb_series(
    FRANCE_10Y_KEY,
    start_period="2026-01",
    end_period="2026-10",
)

germany_10y_current = fetch_ecb_series(
    GERMANY_10Y_KEY,
    start_period="2026-01",
    end_period="2026-10",
)

latest_france_row = (
    france_10y_current
    .sort_values("TIME_PERIOD")
    .iloc[-1]
)

latest_germany_row = (
    germany_10y_current
    .sort_values("TIME_PERIOD")
    .iloc[-1]
)

CURRENT_FRANCE_10Y = (
    latest_france_row["OBS_VALUE"] / 100
)

CURRENT_BUND_10Y = (
    latest_germany_row["OBS_VALUE"] / 100
)

CURRENT_FRANCE_SPREAD_10Y = (
    CURRENT_FRANCE_10Y
    - CURRENT_BUND_10Y
)

print(f"France 10Y: {CURRENT_FRANCE_10Y:.2%}")
print(f"Bund 10Y: {CURRENT_BUND_10Y:.2%}")
print(f"France spread: {CURRENT_FRANCE_SPREAD_10Y:.2%}")

In [ ]:
def build_ar1_expected_path(
    initial_value,
    long_run_mean,
    rho,
    years,
):
    previous = float(initial_value)
    result = {}

    for year in years:
        current = (
            long_run_mean
            + rho
            * (
                previous
                - long_run_mean
            )
        )

        result[year] = current
        previous = current

    return result


CENTRAL_START_YEAR = 2026
CENTRAL_END_YEAR = 2041

CENTRAL_YEARS = list(
    range(
        CENTRAL_START_YEAR,
        CENTRAL_END_YEAR + 1,
    )
)

SIMULATION_YEARS = list(
    range(
        2027,
        CENTRAL_END_YEAR + 1,
    )
)

bund_expected = build_ar1_expected_path(
    initial_value=CURRENT_BUND_10Y,
    long_run_mean=(
        ar1_results[
            "bund_10y"
        ]["long_run_mean"]
    ),
    rho=PERSISTENCE["bund_10y"],
    years=SIMULATION_YEARS,
)

spread_expected = build_ar1_expected_path(
    initial_value=CURRENT_FRANCE_SPREAD_10Y,
    long_run_mean=(
        ar1_results[
            "france_spread_10y"
        ]["long_run_mean"]
    ),
    rho=PERSISTENCE[
        "france_spread_10y"
    ],
    years=SIMULATION_YEARS,
)

In [ ]:
# ============================================================
# IMF 2026 Article IV central macro/fiscal assumptions
# ============================================================

IMF_NOMINAL_GDP_BN = {
    2026: 3056,
    2027: 3137,
    2028: 3236,
    2029: 3334,
    2030: 3436,
    2031: 3537,
}

REAL_GROWTH_CENTRAL = {
    2026: 0.006,
    2027: 0.009,
    2028: 0.012,
    2029: 0.011,
    2030: 0.011,
    2031: 0.011,
}

# CPI is retained as the inflation proxy for linker indexation.
INFLATION_CENTRAL = {
    2026: 0.023,
    2027: 0.017,
    2028: 0.019,
    2029: 0.019,
    2030: 0.019,
    2031: 0.018,
}

# GDP deflator is used for nominal-GDP extrapolation after 2031.
GDP_DEFLATOR_CENTRAL = {
    2026: 0.015,
    2027: 0.017,
    2028: 0.019,
    2029: 0.019,
    2030: 0.019,
    2031: 0.018,
}

PRIMARY_BALANCE_CENTRAL = {
    2026: -0.027,
    2027: -0.023,
    2028: -0.017,
    2029: -0.011,
    2030: -0.006,
    2031: 0.000,
}

OVERALL_BALANCE_CENTRAL = {
    2026: -0.052,
    2027: -0.049,
    2028: -0.045,
    2029: -0.041,
    2030: -0.038,
    2031: -0.035,
}

IMF_PUBLIC_DEBT_GDP = {
    2026: 1.185,
    2027: 1.203,
    2028: 1.211,
    2029: 1.216,
    2030: 1.218,
    2031: 1.218,
}

# Interest expenditure / GDP = primary balance - overall balance.
CENTRAL_INTEREST_GDP_IMF = {
    year: (
        PRIMARY_BALANCE_CENTRAL[year]
        - OVERALL_BALANCE_CENTRAL[year]
    )
    for year in range(2026, 2032)
}

# Post-2031 values are terminal modelling assumptions, not forecasts.
TERMINAL_REAL_GROWTH = REAL_GROWTH_CENTRAL[2031]
TERMINAL_INFLATION = INFLATION_CENTRAL[2031]
TERMINAL_GDP_DEFLATOR = GDP_DEFLATOR_CENTRAL[2031]
TERMINAL_PRIMARY_BALANCE = PRIMARY_BALANCE_CENTRAL[2031]

for year in range(2032, CENTRAL_END_YEAR + 1):
    REAL_GROWTH_CENTRAL[year] = TERMINAL_REAL_GROWTH
    INFLATION_CENTRAL[year] = TERMINAL_INFLATION
    GDP_DEFLATOR_CENTRAL[year] = TERMINAL_GDP_DEFLATOR
    PRIMARY_BALANCE_CENTRAL[year] = TERMINAL_PRIMARY_BALANCE

In [ ]:
# ============================================================
# Assemble central path
# ============================================================

central_path = pd.DataFrame({
    "year": CENTRAL_YEARS,
})

central_path["real_growth"] = (
    central_path["year"].map(REAL_GROWTH_CENTRAL)
)

central_path["inflation"] = (
    central_path["year"].map(INFLATION_CENTRAL)
)

central_path["gdp_deflator"] = (
    central_path["year"].map(GDP_DEFLATOR_CENTRAL)
)

central_path["primary_balance_gdp"] = (
    central_path["year"].map(PRIMARY_BALANCE_CENTRAL)
)

# ------------------------------------------------------------
# Nominal GDP: exact IMF levels through 2031
# ------------------------------------------------------------

central_path["nominal_gdp_eur"] = np.nan

for year, gdp_bn in IMF_NOMINAL_GDP_BN.items():
    central_path.loc[
        central_path["year"] == year,
        "nominal_gdp_eur",
    ] = gdp_bn * 1e9

# Post-2031 nominal GDP from real growth + GDP deflator.
for year in range(2032, CENTRAL_END_YEAR + 1):
    previous_gdp = float(
        central_path.loc[
            central_path["year"] == year - 1,
            "nominal_gdp_eur",
        ].iloc[0]
    )

    nominal_growth = (
        (1 + REAL_GROWTH_CENTRAL[year])
        * (1 + GDP_DEFLATOR_CENTRAL[year])
        - 1
    )

    central_path.loc[
        central_path["year"] == year,
        "nominal_gdp_eur",
    ] = previous_gdp * (1 + nominal_growth)

central_path["nominal_growth"] = (
    central_path["nominal_gdp_eur"].pct_change()
)

central_path.loc[
    central_path["year"] == 2026,
    "nominal_growth",
] = (
    (1 + REAL_GROWTH_CENTRAL[2026])
    * (1 + GDP_DEFLATOR_CENTRAL[2026])
    - 1
)

# ------------------------------------------------------------
# Market central path
# ------------------------------------------------------------

bund_central = {
    2026: CURRENT_BUND_10Y,
    **bund_expected,
}

spread_central = {
    2026: CURRENT_FRANCE_SPREAD_10Y,
    **spread_expected,
}

central_path["bund_10y"] = (
    central_path["year"].map(bund_central)
)

central_path["france_spread_10y"] = (
    central_path["year"].map(spread_central)
)

central_path["france_10y"] = (
    central_path["bund_10y"]
    + central_path["france_spread_10y"]
)

INITIAL_10S30S_SLOPE = (
    BASELINE_OAT_30Y
    - CURRENT_FRANCE_10Y
)

central_path["oat_30y"] = (
    central_path["france_10y"]
    + INITIAL_10S30S_SLOPE
)

central_path["interest_gdp_imf"] = (
    central_path["year"].map(CENTRAL_INTEREST_GDP_IMF)
)

required_central_columns = [
    "real_growth",
    "inflation",
    "gdp_deflator",
    "primary_balance_gdp",
    "nominal_gdp_eur",
    "nominal_growth",
    "bund_10y",
    "france_spread_10y",
    "france_10y",
    "oat_30y",
]

assert not (
    central_path[
        required_central_columns
    ]
    .isna()
    .any()
    .any()
)

assert np.allclose(
    central_path["france_10y"],
    central_path["bund_10y"]
    + central_path["france_spread_10y"],
)

print("Central path validation passed.")
central_path

## 8. Robust stochastic path

In [ ]:
def draw_empirical_innovation(
    rng,
):
    index = rng.integers(
        low=0,
        high=len(
            RESIDUAL_MATRIX
        ),
    )

    return (
        RESIDUAL_MATRIX[
            index
        ].copy()
    )


def generate_calibrated_stochastic_path(
    central_path,
    rng,
    innovation_method="robust_gaussian",
):
    central = (
        central_path
        .set_index("year")
        .copy()
    )

    years = [
        year
        for year in central.index
        if year >= 2027
    ]

    previous = {
        variable:
            float(
                central.loc[
                    2026,
                    variable,
                ]
            )
        for variable
        in CALIBRATION_VARIABLES
    }

    rows = []

    for year in years:
        previous_year = year - 1

        if innovation_method == "robust_gaussian":
            shocks = rng.multivariate_normal(
                mean=np.zeros(len(CALIBRATION_VARIABLES)),
                cov=ROBUST_COVARIANCE_MATRIX,
            )

        elif innovation_method == "gaussian":
            shocks = rng.multivariate_normal(
                mean=np.zeros(
                    len(
                        CALIBRATION_VARIABLES
                    )
                ),
                cov=CALIBRATED_COVARIANCE_MATRIX,
            )

        elif innovation_method == "bootstrap":
            shocks = (
                draw_empirical_innovation(
                    rng
                )
            )

        else:
            raise ValueError(
                "innovation_method must be "
                "'robust_gaussian', 'gaussian', or 'bootstrap'."
            )

        current = {}

        for i, variable in enumerate(
            CALIBRATION_VARIABLES
        ):
            mu_t = float(
                central.loc[
                    year,
                    variable,
                ]
            )

            mu_previous = float(
                central.loc[
                    previous_year,
                    variable,
                ]
            )

            rho = PERSISTENCE[
                variable
            ]

            current[
                variable
            ] = (
                mu_t
                +
                rho
                * (
                    previous[
                        variable
                    ]
                    - mu_previous
                )
                +
                shocks[i]
            )

        france_10y = (
            current[
                "bund_10y"
            ]
            +
            current[
                "france_spread_10y"
            ]
        )

        oat_30y = (
            france_10y
            +
            INITIAL_10S30S_SLOPE
        )

        nominal_growth = (
            (
                1
                + current[
                    "real_growth"
                ]
            )
            *
            (
                1
                + current[
                    "inflation"
                ]
            )
            - 1
        )

        rows.append({
            "year": year,
            **current,
            "france_10y": france_10y,
            "oat_30y": oat_30y,
            "nominal_growth": nominal_growth,
        })

        previous = current

    return pd.DataFrame(rows)

In [ ]:
rng = np.random.default_rng(
    RANDOM_SEED
)

stochastic_path = (
    generate_calibrated_stochastic_path(
        central_path=central_path,
        rng=rng,
        innovation_method="robust_gaussian",
    )
)

stochastic_path

In [ ]:
central_plot = (
    central_path[
        central_path["year"] >= 2027
    ]
    .copy()
    .reset_index(drop=True)
)

plot_specs = [
    ("real_growth", "Real GDP Growth", "%"),
    ("inflation", "Inflation", "%"),
    ("primary_balance_gdp", "Primary Balance / GDP", "% GDP"),
    ("bund_10y", "German 10Y Bund Yield", "%"),
    ("france_spread_10y", "France–Germany 10Y Spread", "%"),
    ("oat_30y", "French 30Y OAT Yield", "%"),
]

fig, axes = plt.subplots(
    3,
    2,
    figsize=(14, 12),
)

axes = axes.flatten()

for ax, (
    column,
    title,
    ylabel,
) in zip(
    axes,
    plot_specs,
):
    ax.plot(
        central_plot["year"],
        central_plot[column] * 100,
        label="Central",
        linewidth=2,
    )

    ax.plot(
        stochastic_path["year"],
        stochastic_path[column] * 100,
        label="Robust stochastic",
        linewidth=1.8,
    )

    ax.axhline(
        0,
        linewidth=0.8,
        alpha=0.3,
    )

    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
    ax.legend()

fig.suptitle(
    "France — Central vs Robust Stochastic Path",
    fontsize=15,
)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 5)
)

ax.plot(
    central_plot["year"],
    central_plot["nominal_growth"] * 100,
    label="Central",
    linewidth=2,
)

ax.plot(
    stochastic_path["year"],
    stochastic_path["nominal_growth"] * 100,
    label="Robust stochastic",
    linewidth=1.8,
)

ax.axhline(
    0,
    linewidth=0.8,
    alpha=0.3,
)

ax.set_title("Nominal GDP Growth")
ax.set_xlabel("Year")
ax.set_ylabel("%")
ax.grid(alpha=0.25)
ax.legend()

plt.show()

## 9. Corrected AFT → general-government bridge

The stochastic AFT path is now compared against the **central AFT path**, not against the old 5.5%-forever sensitivity.

The central general-government path is validated against the IMF debt baseline through 2031.

After 2031, the central GG effective interest rate evolves with changes in the central AFT effective financing rate, scaled by the AFT share of the starting GG debt stock.

The stochastic GG interest bill is:

\[
IC_t^{GG,stochastic}
=
i_t^{GG,central}
D_{t-1}^{GG,stochastic}
+
\Delta IC_t^{AFT}
\]

where:

\[
\Delta IC_t^{AFT}
=
IC_t^{AFT,stochastic}
-
IC_t^{AFT,central}
\]

This removes the reference-scenario mismatch that previously generated negative interest expenditure.

In [ ]:
def build_market_curves_from_path(
    path,
):
    """
    Translate a path containing oat_30y into annual nominal
    and real issuance curves.

    The real-curve transmission remains a first-pass modelling
    assumption: real curves shift by the same amount as the
    30Y nominal OAT relative to the 5.5% reference curve.
    """
    result = {}

    for _, row in path.iterrows():
        year = int(row["year"])
        oat_30y = float(row["oat_30y"])

        nominal_curve = curve_from_30y_anchor(
            oat_30y_yield=oat_30y,
            offsets_to_30y=NOMINAL_CURVE_OFFSETS_TO_30Y,
        )

        yield_shift = (
            oat_30y
            - BASELINE_OAT_30Y
        )

        real_euro_curve = {
            maturity:
                rate + yield_shift
            for maturity, rate
            in baseline_real_euro_curve.items()
        }

        real_fr_curve = {
            maturity:
                rate + yield_shift
            for maturity, rate
            in baseline_real_fr_curve.items()
        }

        result[year] = {
            "nominal": nominal_curve,
            "real_euro": real_euro_curve,
            "real_fr": real_fr_curve,
        }

    return result


central_simulation_path = (
    central_path[
        central_path["year"] >= 2027
    ]
    .copy()
)

stochastic_simulation_path = (
    stochastic_path.copy()
)

central_market_curves = (
    build_market_curves_from_path(
        central_simulation_path
    )
)

stochastic_market_curves = (
    build_market_curves_from_path(
        stochastic_simulation_path
    )
)

In [ ]:
# ============================================================
# Run central and stochastic AFT portfolios
# ============================================================

central_indexed = (
    central_simulation_path.set_index("year")
)

stochastic_indexed = (
    stochastic_simulation_path.set_index("year")
)

aft_central_path, aft_central_final = (
    simulate_portfolio_path(
        initial_portfolio=portfolio,
        initial_gdp_eur=BASELINE["initial_gdp_eur"],
        start_year=BASELINE["start_year"],
        years=BASELINE["horizon_years"],
        nominal_growth=(
            central_indexed[
                "nominal_growth"
            ].to_dict()
        ),
        primary_balance_ratio=(
            central_indexed[
                "primary_balance_gdp"
            ].to_dict()
        ),
        market_curves=central_market_curves,
        issuance_plan=ISSUANCE_PLAN,
        french_inflation=(
            central_indexed[
                "inflation"
            ].to_dict()
        ),
        euro_inflation=(
            central_indexed[
                "inflation"
            ].to_dict()
        ),
    )
)

aft_stochastic_path, aft_stochastic_final = (
    simulate_portfolio_path(
        initial_portfolio=portfolio,
        initial_gdp_eur=BASELINE["initial_gdp_eur"],
        start_year=BASELINE["start_year"],
        years=BASELINE["horizon_years"],
        nominal_growth=(
            stochastic_indexed[
                "nominal_growth"
            ].to_dict()
        ),
        primary_balance_ratio=(
            stochastic_indexed[
                "primary_balance_gdp"
            ].to_dict()
        ),
        market_curves=stochastic_market_curves,
        issuance_plan=ISSUANCE_PLAN,
        french_inflation=(
            stochastic_indexed[
                "inflation"
            ].to_dict()
        ),
        euro_inflation=(
            stochastic_indexed[
                "inflation"
            ].to_dict()
        ),
    )
)

stochastic_aft_cost_shock = pd.Series(
    (
        aft_stochastic_path[
            "total_financing_cost_eur"
        ].to_numpy()
        -
        aft_central_path[
            "total_financing_cost_eur"
        ].to_numpy()
    ),
    index=YEARS,
    name="aft_cost_shock_eur",
)

aft_bridge_diagnostic = pd.DataFrame({
    "year": YEARS,
    "central_aft_cost_bn": (
        aft_central_path[
            "total_financing_cost_eur"
        ].to_numpy()
        / 1e9
    ),
    "stochastic_aft_cost_bn": (
        aft_stochastic_path[
            "total_financing_cost_eur"
        ].to_numpy()
        / 1e9
    ),
    "aft_cost_shock_bn": (
        stochastic_aft_cost_shock.to_numpy()
        / 1e9
    ),
})

aft_bridge_diagnostic

In [ ]:
# ============================================================
# Build IMF-consistent central general-government path
# ============================================================

AFT_SHARE_OF_INITIAL_GG_DEBT = min(
    1.0,
    total_debt(portfolio)
    / INITIAL_GG_DEBT_EUR,
)

print(
    f"AFT indexed debt / initial GG debt: "
    f"{AFT_SHARE_OF_INITIAL_GG_DEBT:.1%}"
)


def build_central_gg_path(
    central_path,
    aft_central_path,
):
    """
    2027–2031:
        Use IMF nominal GDP, primary balance and implied
        interest/GDP directly.

    2032 onward:
        Keep the central macro/fiscal path and evolve the
        GG effective interest rate with changes in the central
        AFT effective financing rate, scaled by the AFT share
        of the initial GG debt stock.
    """

    central = central_path.set_index("year")
    aft = aft_central_path.set_index("year")

    debt = INITIAL_GG_DEBT_EUR
    previous_effective_rate = None
    rows = []

    for year in YEARS:
        gdp_end = float(
            central.loc[
                year,
                "nominal_gdp_eur",
            ]
        )

        pb_ratio = float(
            central.loc[
                year,
                "primary_balance_gdp",
            ]
        )

        pb_eur = (
            pb_ratio
            * gdp_end
        )

        if year <= 2031:
            interest_gdp = float(
                central.loc[
                    year,
                    "interest_gdp_imf",
                ]
            )

            interest_eur = (
                interest_gdp
                * gdp_end
            )

            effective_rate = (
                interest_eur
                / debt
            )

        else:
            aft_rate_change = (
                float(
                    aft.loc[
                        year,
                        "effective_financing_rate",
                    ]
                )
                -
                float(
                    aft.loc[
                        year - 1,
                        "effective_financing_rate",
                    ]
                )
            )

            effective_rate = (
                previous_effective_rate
                +
                AFT_SHARE_OF_INITIAL_GG_DEBT
                * aft_rate_change
            )

            if effective_rate <= 0:
                raise ValueError(
                    f"Central GG effective rate became "
                    f"non-positive in {year}: "
                    f"{effective_rate:.4%}"
                )

            interest_eur = (
                effective_rate
                * debt
            )

            interest_gdp = (
                interest_eur
                / gdp_end
            )

        debt_end = (
            debt
            + interest_eur
            - pb_eur
        )

        rows.append({
            "year": year,
            "gdp_end_eur": gdp_end,
            "debt_start_eur": debt,
            "primary_balance_gdp": pb_ratio,
            "primary_balance_eur": pb_eur,
            "interest_eur": interest_eur,
            "interest_gdp": interest_gdp,
            "effective_interest_rate": effective_rate,
            "debt_end_eur": debt_end,
            "debt_gdp_end": (
                debt_end
                / gdp_end
            ),
        })

        debt = debt_end
        previous_effective_rate = effective_rate

    return pd.DataFrame(rows)


gg_central_path = (
    build_central_gg_path(
        central_path=central_path,
        aft_central_path=aft_central_path,
    )
)


# ============================================================
# Validate central GG path against IMF through 2031
# ============================================================

imf_validation = (
    gg_central_path[
        gg_central_path["year"] <= 2031
    ][
        [
            "year",
            "debt_gdp_end",
            "interest_gdp",
        ]
    ]
    .copy()
)

imf_validation["imf_debt_gdp"] = (
    imf_validation["year"]
    .map(IMF_PUBLIC_DEBT_GDP)
)

imf_validation["debt_error_pp"] = (
    (
        imf_validation[
            "debt_gdp_end"
        ]
        -
        imf_validation[
            "imf_debt_gdp"
        ]
    )
    * 100
)

print(
    imf_validation.to_string(
        index=False
    )
)

assert (
    imf_validation[
        "debt_error_pp"
    ]
    .abs()
    .max()
    < 0.15
), (
    "Central GG path does not reproduce the "
    "IMF debt baseline closely enough."
)

print("IMF central-path validation passed.")


# ============================================================
# Stochastic general-government DSA
# ============================================================

def simulate_stochastic_gg_dsa(
    stochastic_path,
    gg_central_path,
    aft_cost_shock,
):
    """
    Apply the central GG effective rate to the stochastic debt
    stock, then add only the incremental AFT financing-cost
    deviation from the central AFT path.
    """

    stochastic = stochastic_path.set_index("year")
    central_gg = gg_central_path.set_index("year")

    debt = INITIAL_GG_DEBT_EUR
    gdp = BASELINE["initial_gdp_eur"]

    rows = []

    for year in YEARS:
        nominal_growth = float(
            stochastic.loc[
                year,
                "nominal_growth",
            ]
        )

        gdp_end = (
            gdp
            * (1 + nominal_growth)
        )

        pb_ratio = float(
            stochastic.loc[
                year,
                "primary_balance_gdp",
            ]
        )

        pb_eur = (
            pb_ratio
            * gdp_end
        )

        central_effective_rate = float(
            central_gg.loc[
                year,
                "effective_interest_rate",
            ]
        )

        baseline_interest_on_stock = (
            central_effective_rate
            * debt
        )

        aft_shock_eur = float(
            aft_cost_shock.loc[year]
        )

        total_interest_eur = (
            baseline_interest_on_stock
            + aft_shock_eur
        )

        if total_interest_eur <= 0:
            raise ValueError(
                f"Negative/non-positive GG interest bill "
                f"in {year}: "
                f"€{total_interest_eur / 1e9:.2f}bn"
            )

        implied_effective_rate = (
            total_interest_eur
            / debt
        )

        debt_end = (
            debt
            + total_interest_eur
            - pb_eur
        )

        rows.append({
            "year": year,
            "gdp_start_eur": gdp,
            "gdp_end_eur": gdp_end,
            "nominal_growth": nominal_growth,
            "debt_start_eur": debt,
            "central_effective_rate": central_effective_rate,
            "aft_interest_cost_shock_eur": aft_shock_eur,
            "total_interest_eur": total_interest_eur,
            "implied_effective_rate": implied_effective_rate,
            "interest_gdp": (
                total_interest_eur
                / gdp_end
            ),
            "primary_balance_gdp": pb_ratio,
            "primary_balance_eur": pb_eur,
            "debt_end_eur": debt_end,
            "debt_gdp_end": (
                debt_end
                / gdp_end
            ),
        })

        gdp = gdp_end
        debt = debt_end

    return pd.DataFrame(rows)


gg_stochastic_path = (
    simulate_stochastic_gg_dsa(
        stochastic_path=stochastic_path,
        gg_central_path=gg_central_path,
        aft_cost_shock=stochastic_aft_cost_shock,
    )
)

gg_stochastic_path[
    [
        "year",
        "nominal_growth",
        "primary_balance_gdp",
        "central_effective_rate",
        "implied_effective_rate",
        "interest_gdp",
        "debt_gdp_end",
    ]
]

In [ ]:
# ============================================================
# Central vs stochastic GG debt
# ============================================================

fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.plot(
    gg_central_path["year"],
    gg_central_path["debt_gdp_end"] * 100,
    label="Central IMF-consistent path",
)

ax.plot(
    gg_stochastic_path["year"],
    gg_stochastic_path["debt_gdp_end"] * 100,
    label="Robust stochastic path",
)

ax.set_title(
    "French General-Government Debt — Central vs Stochastic"
)

ax.set_xlabel("Year")
ax.set_ylabel("Debt / GDP (%)")
ax.legend()
ax.grid(alpha=0.25)

plt.show()


# ============================================================
# Accounting diagnostics
# ============================================================

debt_diagnostic = pd.DataFrame({
    "year": YEARS,
    "central_debt_gdp": (
        gg_central_path[
            "debt_gdp_end"
        ].to_numpy()
    ),
    "stochastic_debt_gdp": (
        gg_stochastic_path[
            "debt_gdp_end"
        ].to_numpy()
    ),
    "stochastic_nominal_growth": (
        gg_stochastic_path[
            "nominal_growth"
        ].to_numpy()
    ),
    "stochastic_primary_balance_gdp": (
        gg_stochastic_path[
            "primary_balance_gdp"
        ].to_numpy()
    ),
    "stochastic_interest_gdp": (
        gg_stochastic_path[
            "interest_gdp"
        ].to_numpy()
    ),
    "aft_cost_shock_bn": (
        stochastic_aft_cost_shock.to_numpy()
        / 1e9
    ),
})

debt_diagnostic["debt_gap_pp"] = (
    (
        debt_diagnostic[
            "stochastic_debt_gdp"
        ]
        -
        debt_diagnostic[
            "central_debt_gdp"
        ]
    )
    * 100
)

display_diag = debt_diagnostic.copy()

for col in [
    "central_debt_gdp",
    "stochastic_debt_gdp",
    "stochastic_nominal_growth",
    "stochastic_primary_balance_gdp",
    "stochastic_interest_gdp",
]:
    display_diag[col] *= 100

display_diag.round(2)

# Current stopping point

This corrected version fixes the reference-scenario mismatch that previously drove general-government interest expenditure below zero.

The model now has:

- an IMF-consistent 2026 starting point;
- exact IMF nominal GDP, fiscal balance, primary balance and debt targets through 2031;
- nominal GDP growth based on the IMF nominal-GDP path rather than HICP;
- a central AFT refinancing path;
- a robust stochastic AFT path;
- AFT financing-cost shocks measured **relative to the central AFT path**;
- a central GG debt path validated against IMF 2027–2031 debt ratios;
- post-2031 central GG interest-rate evolution linked to central AFT repricing;
- stochastic GG interest costs that preserve debt-stock feedback;
- a hard failure if the model ever produces a non-positive general-government interest bill.

The next stage is to validate the corrected single stochastic path and then scale the model to Monte Carlo.